# Evaluation — Retrieval Quality and Hallucination Analysis

## Goal
A RAG system is only as good as its retrieval.
This notebook measures:
- Retrieval quality: did we get the right chunks?
- Hallucination: did the model answer from context or make things up?

## Key Questions
1. When we ask about topic X, do we retrieve chunks about topic X?
2. Does the answer contain information not in the retrieved context?

In [12]:
import re
import numpy as np
import faiss
import requests
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Setup - Rebuild Knowledge Base

In [8]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

sample_document = """
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed. 
It focuses on developing computer programs that can access data and use it 
to learn for themselves.

The process begins with observations or data, such as examples, direct 
experience, or instruction. Machine learning algorithms use computational 
methods to learn information directly from data without relying on a 
predetermined equation as a model.

Deep learning is a subfield of machine learning that uses neural networks 
with many layers. These deep neural networks have revolutionized fields such 
as computer vision, natural language processing, and speech recognition.

Natural language processing (NLP) is another important area of machine learning. 
It enables computers to understand, interpret, and generate human language. 
Applications include sentiment analysis, machine translation, and chatbots.

Reinforcement learning is a type of machine learning where an agent learns 
to make decisions by interacting with an environment. The agent receives 
rewards for correct actions and penalties for incorrect ones.
"""

def clean_text(text: str) -> str:
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r" +", " ", text)
    return text.strip()

def chunk_by_sentences(text: str, sentences_per_chunk: int = 3) -> list:
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = " ".join(sentences[i:i + sentences_per_chunk])
        if chunk.strip():
            chunks.append(chunk.strip())
    return chunks

cleaned_doc = clean_text(sample_document)
chunks = chunk_by_sentences(cleaned_doc)
chunk_embeddings = embed_model.encode(chunks).astype(np.float32)
index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

print(f"Knowledge base ready. Chunks: {len(chunks)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base ready. Chunks: 4


## 2. Retrieval Quality Evaluation
For each query, we know which chunk should be retrieved.
We measure if the correct chunk is ranked first.

In [9]:
# Ground truth — query to correct chunk index
ground_truth = [
    {"query": "What is deep learning?", "correct_chunk": 1},
    {"query": "How does reinforcement learning work?", "correct_chunk": 3},
    {"query": "What are NLP applications?", "correct_chunk": 2},
    {"query": "What is machine learning?", "correct_chunk": 0},
]

def retrieve(query: str, top_k: int = 3) -> list:
    """Retrieve top-k chunks with indices and distances."""
    query_embedding = embed_model.encode([query]).astype(np.float32)
    distances, indices = index.search(query_embedding, top_k)
    return list(indices[0])

# Evaluate
correct = 0
print(f"{"Query":<45} {"Expected":>8} {"Got":>5} {"Hit":>5}")
print("-" * 65)

for item in ground_truth:
    retrieved = retrieve(item["query"])
    top1 = retrieved[0]
    hit = top1 == item["correct_chunk"]
    if hit:
        correct += 1
    print(f"{item["query"]:<45} {item["correct_chunk"]:>8} {top1:>5} {'✓' if hit else "x":>5}")

print(f"\nRetrieval Accuracy: {correct}/{len(ground_truth)} = {correct/len(ground_truth):.0%}")

Query                                         Expected   Got   Hit
-----------------------------------------------------------------
What is deep learning?                               1     1     ✓
How does reinforcement learning work?                3     3     ✓
What are NLP applications?                           2     2     ✓
What is machine learning?                            0     0     ✓

Retrieval Accuracy: 4/4 = 100%


## 3. Hallucination Analysis
Does the answer contain information not present in the retrieved context?
We measure this by checking semantic similarity between answer and context.

In [17]:
def generate(query: str, context: list) -> str:
    context_text = "\n".join(context)
    prompt = f"""Answer the question based only on the context below.
Context:
{context_text}

Question: {query}
Answer:"""
    
    response = requests.post(
        "http://127.0.0.1:11434/api/chat",
        json={
            "model": "gemma3:12b",
            "messages": [{"role": "user", "content": prompt}],
            "stream": False
        },
        proxies={"http": None, "https": None}
    )
    return response.json()["message"]["content"]


def hallucination_score(answer: str, context: list) -> float:
    """
    Measure how grounded the answer is in the context.
    Score close to 1.0 = answer is grounded in context.
    Score close to 0.0 = answer may be hallucinated.
    """
    answer_embedding = embed_model.encode([answer])
    context_embedding = embed_model.encode([' '.join(context)])
    score = cosine_similarity(answer_embedding, context_embedding)[0][0]
    return float(score)


# Test
test_queries = [
    {"query": "What is deep learning?", "context_idx": [1, 0]},
    {"query": "How does reinforcement learning work?", "context_idx": [3, 0]},
    {"query": "What are NLP applications?", "context_idx": [2, 1]},
]

print(f"{"Query":<45} {"Grounding Score":>15} {"Status":>10}")
print("-" * 72)

for item in test_queries:
    context = [chunks[i] for i in item["context_idx"]]
    answer = generate(item["query"], context)
    score = hallucination_score(answer, context)
    status = "Grounded" if score > 0.5 else "Possible Hallucination"
    print(f"{item["query"]:<45} {score:>15.4f} {status:>10}")

Query                                         Grounding Score     Status
------------------------------------------------------------------------
What is deep learning?                                 0.8075   Grounded
How does reinforcement learning work?                  0.8144   Grounded
What are NLP applications?                             0.6629   Grounded


## 4. Key Observations

| Query | Grounding Score | Status |
|-------|----------------|--------|
| Deep learning | 0.81 | Grounded |
| Reinforcement learning | 0.81 | Grounded |
| NLP applications | 0.66 | Grounded |

## What Grounding Score Means
- Score > 0.5: answer is semantically similar to context — grounded
- Score < 0.5: answer may contain information not in context — hallucination

## Key Insight
All answers scored above 0.66 — the model answered from context, not from memory.
NLP applications scored lower (0.66) because the answer was very concise
("Sentiment analysis, machine translation, and chatbots")
while the context contained more information.

## Retrieval Quality Summary
- Retrieval accuracy: 100% — correct chunk retrieved for every query
- Grounding score: 0.66 - 0.81 — all answers grounded in context

## Conclusion
This RAG system successfully:
1. Ingests and chunks documents
2. Retrieves relevant chunks using semantic search
3. Generates grounded answers from context
4. Evaluates retrieval quality and hallucination